## Dataset 
contains detailed product, customers, orders and review from ecommerce website.

## Scope: 
This data will be used to build dashboards that help managers and salespeople make business decisions.

## Dimension modelling: 
I chose the Star Schema for this analytics layer as it is a common best practice, especially in BI and reporting scenarios, because it optimizes query performance, simplicity, and usability.


Order_items --> Fact tabel


## Dimension tabels and their keys:

order_items.order_id → orders.order_id

orders.customer_id → customers.customer_id

order_items.product_id → products.product_id

products.category_id → categories.category_id

review.customer_id → customers.customer_id

## 1. Loading Raw CSV Data into the Landing Layer
This cell reads raw CSV files for each entity in the ecommerce dataset (order_items, orders, categories, customers, products, reviews)
- into Spark DataFrames with schema inference and headers enabled. It then writes each DataFrame as a Delta table
- into the 'raw_ecommerce' database. This process establishes the raw (landing) layer for further data processing,
- ensuring the data is available in an optimized, queryable format for downstream analytics and ETL pipelines.

In [0]:
df_order_items = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/FileStore/tables/order_items.csv")

df_order_items.write.format("delta").mode("overwrite").saveAsTable("raw_ecommerce.raw_order_items")

###################################################################################

df_orders = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("dbfs:/FileStore/tables/orders.csv")

df_orders.write.format("delta").mode("overwrite").saveAsTable("raw_ecommerce.raw_orders")

###################################################################################

df_categories = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/FileStore/tables/categories.csv")

df_categories.write.format("delta").mode("overwrite").saveAsTable("raw_ecommerce.raw_categories")

###################################################################################

df_customers = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/FileStore/tables/customers.csv")

df_customers.write.format("delta").mode("overwrite").saveAsTable("raw_ecommerce.raw_customers")

###################################################################################

df_products = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/FileStore/tables/products.csv")

df_products.write.format("delta").mode("overwrite").saveAsTable("raw_ecommerce.raw_products")

###################################################################################

df_reviews = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/FileStore/tables/reviews.csv")

df_reviews.write.format("delta").mode("overwrite").saveAsTable("raw_ecommerce.raw_reviews")

## 2. Checking for NULLs in Primary Key Columns
- This cell counts the number of rows with NULLs in the primary key columns for each raw ecommerce table.
- This is important for data quality checks, as primary keys should not be NULL in dimension and fact tables.

In [0]:

df_order_items.filter(df_order_items.order_id.isNull()).count()
print(f"Total null rows for key in schema order_items is: {df_order_items.filter(df_order_items.order_id.isNull()).count()}")

df_orders.filter(df_orders.customer_id.isNull()).count()
print(f"Total null rows for key in schema orders is: {df_orders.filter(df_orders.customer_id.isNull()).count()}")

df_categories.filter(df_categories.category_id.isNull()).count()
print(f"Total null rows for key in schema categories is: {df_categories.filter(df_categories.category_id.isNull()).count()}")

df_customers.filter(df_customers.customer_id.isNull()).count()
print(f"Total null rows for key in schema customers is:{df_customers.filter(df_customers.customer_id.isNull()).count()}")

df_products.filter(df_products.product_id .isNull()).count()
print(f"Total null rows for key in schema products is:{df_products.filter(df_products.product_id.isNull()).count()}")

df_reviews.filter(df_reviews.review_id.isNull()).count()
print(f"Total null rows for key in schema reviews is:{df_reviews.filter(df_reviews.review_id.isNull()).count()}")


## 3. Writing Raw DataFrames to the Bronze Layer
- This cell writes the previously loaded raw DataFrames into the 'bronze_ecommerce' database as Delta tables.
- This step creates the bronze (curated) layer, which serves as a clean, queryable foundation for further ETL and analytics.
- Using Delta format ensures ACID transactions, scalable metadata handling, and efficient data management.

In [0]:
df_order_items.write.format("delta").mode("overwrite").saveAsTable("bronze_ecommerce.bronze_order_items")

df_orders.write.format("delta").mode("overwrite").saveAsTable("bronze_ecommerce.bronze_orders")

df_categories.write.format("delta").mode("overwrite").saveAsTable("bronze_ecommerce.bronze_categories")

df_customers.write.format("delta").mode("overwrite").saveAsTable("bronze_ecommerce.bronze_customers")

df_products.write.format("delta").mode("overwrite").saveAsTable("bronze_ecommerce.bronze_products")

df_reviews.write.format("delta").mode("overwrite").saveAsTable("bronze_ecommerce.bronze_reviews")